In [5]:
import pandas as pd
import yaml
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from pathlib import Path
from datetime import datetime

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

In [3]:
# Vegetation zonal stats output
zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run__20260224/'
parquet_name = f'veg_model_zonal_stats_v{cn.veg_model_version_underscore}_20260224_19_21_07.parquet'

In [4]:
%%time

# Reads gross outputs parquet table
df = pd.read_parquet(f'{zonal_stats_folder}{parquet_name}')
df["WDPA_high_protection"] = df["WDPA"].isin([1, 2, 3]).astype(int)
df["country_name"] = df["country_name"].replace({
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Russian Federation": "Russia",
    "Democratic Republic of the Congo": "DR Congo",
    "United States of America (the)": "USA"
})

print(f"Rows in df: {len(df)}")
# df

Rows in df: 27999047
CPU times: user 31.6 s, sys: 32.1 s, total: 1min 3s
Wall time: 21.2 s


In [6]:
# QC for df-- to check that all columns have expected options for values
# Not worried about "IOPub data rate exceeded." warnings.

print(f"Columns in df are: {df.columns}\n")
for column in df.columns:
    col_vals = (
        df[column]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )
    print(f"{column} ({len(col_vals)} values): {col_vals}\n")

Columns in df are: Index(['analysis_layer', 'adm0', 'land_state_node', 'WDPA', 'cont_eco',
       'Landmark', 'starting_composite_primary_forest', 'year', 'value',
       'tile_id', 'area_ha', 'land_state_meaning', 'land_state_broad_class',
       'land_state_detailed_class', 'country_name', 'region', 'continent',
       'continent_ecozone', 'WDPA_type', 'density__Mg_ha',
       'WDPA_high_protection'],
      dtype='object')

analysis_layer (21 values): ['carbon_density__non_soil__MgC_ha', 'gross_emissions__AGC__MgCO2', 'gross_emissions__BGC__MgCO2', 'gross_emissions__CH4__MgCO2e', 'gross_emissions__N2O__MgCO2e', 'gross_emissions__all_C_pools__CO2_only__MgCO2', 'gross_emissions__all_C_pools__all_gases__MgCO2e', 'gross_emissions__all_C_pools__non_CO2_only__MgCO2e', 'gross_emissions__deadwood_C__MgCO2', 'gross_emissions__litter_C__MgCO2', 'gross_removals__AGC__MgCO2', 'gross_removals__BGC__MgCO2', 'gross_removals__all_C_pools__MgCO2', 'gross_removals__deadwood_C__MgCO2', 'gross_removals_

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



tile_id (293 values): ['00N_000E', '00N_010E', '00N_020E', '00N_030E', '00N_040E', '00N_040W', '00N_050E', '00N_050W', '00N_060W', '00N_070E', '00N_070W', '00N_080W', '00N_090E', '00N_090W', '00N_100E', '00N_100W', '00N_110E', '00N_120E', '00N_130E', '00N_140E', '00N_150E', '00N_160E', '00N_170E', '00N_180W', '10N_000E', '10N_010E', '10N_010W', '10N_020E', '10N_020W', '10N_030E', '10N_040E', '10N_050E', '10N_050W', '10N_060W', '10N_070E', '10N_070W', '10N_080E', '10N_080W', '10N_090E', '10N_090W', '10N_100E', '10N_100W', '10N_110E', '10N_120E', '10N_130E', '10N_150E', '10N_160E', '10N_170E', '10S_010E', '10S_020E', '10S_030E', '10S_040E', '10S_040W', '10S_050E', '10S_050W', '10S_060W', '10S_070W', '10S_080W', '10S_110E', '10S_120E', '10S_130E', '10S_140E', '10S_150E', '10S_150W', '10S_160E', '10S_160W', '10S_170E', '10S_180W', '20N_000E', '20N_010E', '20N_010W', '20N_020E', '20N_020W', '20N_030E', '20N_030W', '20N_040E', '20N_050E', '20N_060W', '20N_070E', '20N_070W', '20N_080E', '20N_

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



land_state_meaning (82 values): ['Annual cropland converted to anything else with fire', 'Annual cropland converted to anything else without fire', 'Annual cropland converted to short vegetation with fire', 'Annual cropland converted to short vegetation without fire', 'Annual cropland converted to water', 'Cropland gain', 'Cropland remaining cropland with fire', 'Cropland remaining cropland without fire', 'Forest partially disturbed in the current interval without signif. height increase after with fire', 'Forest partially disturbed in the current interval without signif. height increase after without fire', 'Full loss of non-oil palm planted forest as cropland with fire', 'Full loss of non-oil palm planted forest as cropland without fire', 'Full loss of non-oil palm planted forest as short vegetation with fire', 'Full loss of non-oil palm planted forest as short vegetation without fire', 'Full loss of non-oil palm planted forest to anything else without fire', 'Full loss of non-oil pa

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



WDPA_high_protection (2 values): [0, 1]



In [ ]:
# Counts NAs in df. 
# There shouldn't be any. 

df.isna().sum()

Create simplified/abbreviated table that can be used in PyGWalker

In [10]:
%%time

# To create a wide-format table (with fluxes only, not areas of flux densities).
# Drops a few contextual columns to reduce the number of rows
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

id_cols_by_land_state = [
    # "adm0",
    # "country_name",
    # "region",
    # "land_state_node",
    # "land_state_meaning",
    # "land_state_broad_class",
    # "land_state_detailed_class",
    # "WDPA",
    # "WDPA_type",
    # "cont_eco",
    # "continent",
    # "continent_ecozone",
    # "Landmark",
    # "starting_composite_primary_forest",
    "year",
    "tile_id",
    # "area_ha",
    # "density__Mg_ha",
]

df_wide_by_land_state = (
    df.groupby(id_cols_by_land_state + ["analysis_layer"], dropna=False)["value"]
      .sum()
      .unstack("analysis_layer")
      .reset_index()
)

df_wide_by_land_state.to_csv('/mnt/c/GIS/veg_1_0_5_zonal_stats_by_tile__20260531.csv')

CPU times: user 3.11 s, sys: 608 ms, total: 3.72 s
Wall time: 3.77 s


In [ ]:
%%time
# Number of options for each contextual column

summary = {
    col: df[col].nunique()
    for col in df.columns
    if col not in ["index", "value", "analysis_layer", "area_ha", "density__Mg_ha"]
}

pd.Series(summary).sort_values(ascending=False)

In [ ]:
%%time

layers_to_drop = [
    "carbon_density__non_soil__MgC_ha",
    cn.agc_gross_emis_pattern,
    cn.bgc_gross_emis_pattern,
    cn.deadwood_c_gross_emis_pattern,
    cn.litter_c_gross_emis_pattern,
    cn.net_flux_all_C_pools_CO2_only_pattern,
    cn.agc_gross_removals_pattern,
    cn.bgc_gross_removals_pattern,
    cn.deadwood_c_gross_removals_pattern,
    cn.litter_c_gross_removals_pattern,
    cn.net_flux_agc_pattern,
    cn.net_flux_bgc_pattern,
    cn.net_flux_deadwood_c_pattern,
    cn.net_flux_litter_c_pattern,
    cn.ch4_gross_emis_pattern,
    cn.n2o_gross_emis_pattern
]

df_outputs_dropped = df[~df["analysis_layer"].isin(layers_to_drop)].reset_index(drop=True)
# df_outputs_dropped

In [ ]:
%%time
# Drops contextual layers sequentially to get a sense of how many combination rows are added by including each one,
# i.e. how many rows are lost when I drop that one column, given that all the other columns are still present.
# per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/699f45bf-932c-8327-95dc-12ded8c246f5

# Contextual layers that are essentially text versions of others, so they need to be dropped in order to assess how much complexity each contextual layer gives 
# (since they are redundant in terms of complexity)
df_by_context = df_outputs_dropped.drop(columns=['country_name', 'land_state_meaning', 'continent_ecozone', 'WDPA_type', 'region', 'continent', 'land_state_detailed_class', 'land_state_broad_class'])

base = len(df_by_context)
print(f"Rows in gross_emissions__all_C_pools__CO2_only__MgCO2 with all contextual layers: {base}")

cols_to_check = ['tile_id', 'adm0', 'cont_eco', 'WDPA', 'year', 'Landmark', 'starting_composite_primary_forest', 'WDPA_high_protection']

for col in cols_to_check:

    # Drops specified contextual layers to reduce the number of rows in the table (although dropping adm0 or land_state seems to remove only a few 10s of thousands of rows)
    cols_to_sum = ["value", "area_ha", "density__Mg_ha"]
    
    group_cols = [
        c for c in df_by_context.columns
        if c not in [col] + cols_to_sum  # Contextual columns to drop
    ]
    
    df_agg = (
        df_by_context.groupby(group_cols, dropna=False)
          .size()
          .reset_index()
    )
    reduced=len(df_agg)

    print(f"{col:35s} removing it reduces rows by {base - reduced}")

In [ ]:
# Number of rows for each analysis layer 
df_outputs_dropped["analysis_layer"].value_counts()

In [ ]:
# Drops various contextual layers to reduce df size
print(f"Columns before dropping: {df_outputs_dropped.columns}")
# Comment out the contextual layers to keep. 
# Layers on the same line are redundant with each other; they need to be dropped or retained together for full effect.
contextual_layers_to_drop = [   
                             'tile_id', 
                             # cn.adm0_pattern, 'country_name', 'region', 
                             cn.WDPA_pattern, 'WDPA_type', 
                             'WDPA_high_protection',
                             # cn.cont_eco_zstats_pattern, 'continent_ecozone', 'continent', 
                             cn.landmark_pattern, 
                             # cn.starting_composite_primary_forest_pattern,
                             # cn.land_state_pattern, 'land_state_meaning', 
                             # 'land_state_broad_class', 
                             # 'land_state_detailed_class'
                            ]
df_outputs_context_dropped = df_outputs_dropped.drop(columns=contextual_layers_to_drop)

base = len(df_outputs_context_dropped)
print(f"Rows in df with outputs removed, with all contextual layers: {base}")

# Drops specified contextual layers to reduce the number of rows in the table (although dropping adm0 or land_state seems to remove only a few 10s of thousands of rows)
cols_to_sum = ["value", "area_ha", "density__Mg_ha"]

group_cols = [
    c for c in df_outputs_context_dropped.columns
    if c not in contextual_layers_to_drop + cols_to_sum  # Contextual columns to drop
]
# print("group_cols:", group_cols)

df_outputs_context_dropped_agg = (
    df_outputs_context_dropped.groupby(group_cols, dropna=False)
      .sum(numeric_only=True)
      .reset_index()
)
reduced=len(df_outputs_context_dropped_agg)

print(f"Columns after dropping: {df_outputs_context_dropped_agg.columns}")
print(f"Removing tile_id reduces rows by {base - reduced}")
print(f"Rows in output with contextual layers dropped: {reduced}")

In [ ]:
%%time

# Test sums for original and simplified tables. Values should be identical.
year = 2024
variable = cn.net_flux_all_C_pools_all_gases_pattern
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")

year = 2020
variable = cn.gross_removals_all_C_pools_pattern
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")

year = 2018
variable = cn.gross_emis_all_C_pools_CO2_only_pattern
print(f"Full df:        {int(df[(df['year'] == year) & (df['analysis_layer'] ==variable)]['value'].sum())}")
print(f"Simplified df:  {int(df_outputs_context_dropped_agg[(df_outputs_context_dropped_agg['year'] == year) & (df_outputs_context_dropped_agg['analysis_layer']==variable)]['value'].sum())}")

In [ ]:
# # Way too large to work in PyGWalker (28 million rows)
# walker = pyg.walk(df)

In [ ]:
# # Way too large to work in PyGWalker (8.2 million rows)
# walker = pyg.walk(df_outputs_dropped)

In [ ]:
# Loads in PyGWalker (1289471 rows works. NOTE: 1.8 million rows worked for a bit and then crashed)
walker = pyg.walk(df_outputs_context_dropped_agg)

In [ ]:
# other_full_df = pd.read_parquet(f'{veg_model_parquet_folder}{parquet_core_name}other_outputs_1x1.parquet')
# primary_2015_df = other_full_df[other_full_df['layer_name'] == 'composite_primary_forest_2015']
# primary_2015_df.head()
# primary_2015_df.to_csv(f'/mnt/c/GIS/primary_2015.csv', index=False)

In [ ]:
# # Exports to a csv so data can be used in Excel or reused
# gross_net_aggreg_df.to_csv(f'/mnt/c/GIS/global_iso_aggreg_v{model_version}.csv', index=False)

# gross_net_full_wide_df = gross_net_full_df.pivot(
#     index=['chunk_id', 'years', 'iso'],
#     columns='pattern',
#     values='sum_value'
# ).reset_index()
# gross_net_full_wide_df
# gross_net_full_wide_df.to_csv(f'/mnt/c/GIS/global_chunk_v_{model_version}_wide.csv', index=False)

In [ ]:
# # Groups and sums outputs by country 
# gross_net_aggreg_df = gross_net_full_df.groupby(['pattern', 'years', 'iso', 'tropical'], as_index=False)['sum_value'].sum()
# # gross_net_aggreg_df

In [ ]:
# gross_net_df_example_ISO = gross_net_full_df[gross_net_full_df['iso'] == 'RUS']
# # gross_net_df_RUS

In [ ]:
# gross_net_df_example_chunk = gross_net_full_df[gross_net_full_df['chunk_id'] == '124_-25_125_-24']

In [ ]:
# pyg.walk(gross_net_df_example_ISO, spec=vis_spec)